In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [2]:
#Step 3 Create the Model
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=groq_api_key
)

c:\DS2026\Python\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#Step 4 Send a Message
from langchain_core.messages import HumanMessage

response = model.invoke(
    [
        HumanMessage(content="Hi, My name is Bensly")
    ]
)

print(response.content)

Hello Bensly! It's nice to meet you. Is there something I can help you with or would you like to chat?


In [5]:
#Step 5 Multiple Messages
from langchain_core.messages import AIMessage

response = model.invoke([
    HumanMessage(content="Hi, My name is Bensly"),
    AIMessage(content="Hello Bensly"),
    HumanMessage(content="What's my name?")
    ])

In [6]:
#Step 6 ChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

C:\Users\bensl\AppData\Local\Temp\ipykernel_13908\2879068034.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory


In [7]:
#Step 7 Store Conversation
store = {}

In [8]:
#Step 8 Create Session Function
from langchain_core.chat_history import BaseChatMessageHistory

def get_session_history(session_id:str)->BaseChatMessageHistory:

    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    return store[session_id]

In [9]:
#Step 9 RunnableWithMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

with_message_history = RunnableWithMessageHistory(
    model,
    get_session_history
)

c:\DS2026\Python\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3701: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [10]:
#Step 10 Session ID
config = {
    "configurable":{
        "session_id":"chat1"
    }
}

In [11]:
#Step 11 First Conversation
response = with_message_history.invoke(
    [HumanMessage(content="Hi My name is Bensly")],
    config=config
)

In [12]:
#Step 12 Next Question
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config
)

In [13]:
#Step 13 Different User
config = {
    "configurable":{
        "session_id":"chat2"
    }
}

Prompt Templates

Instead of sending only,we can add instructions.

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder

In [15]:
prompt = ChatPromptTemplate.from_messages(
[
    (
        "system",
        "You are a helpful assistant."
    ),

    MessagesPlaceholder(variable_name="messages")
]
)

In [16]:
MessagesPlaceholder("messages")

MessagesPlaceholder(variable_name='messages')

In [17]:
chain = prompt | model

In [18]:
chain.invoke(
{
    "messages":[HumanMessage(content="Hello")],
    "language":"Hindi"
}
)

AIMessage(content='Hello. How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 42, 'total_tokens': 52, 'completion_time': 0.030234899, 'completion_tokens_details': None, 'prompt_time': 0.00119129, 'prompt_tokens_details': None, 'queue_time': 0.054666399, 'total_time': 0.031426189}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fb5f3-eaaa-7532-b422-64879fec8221-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 10, 'total_tokens': 52})

In [19]:
RunnableWithMessageHistory(

    chain,

    get_session_history,

    input_messages_key="messages"
)

c:\DS2026\Python\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3701: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  messages: RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
}), kwargs={}, config={'run_name': 'insert_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function get_session_history at 0x000001B853223420>, input_messages_key='messages', history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

Conversation History Problem

Suppose after 6 months.

History becomes

5000 messages

LLMs have limited context windows.

Eventually

Token Limit Exceeded

or

Very expensive requests.

trim_messages()

In [20]:
from langchain_core.messages import trim_messages

In [21]:
trimmer = trim_messages(

    max_tokens=45,

    strategy="last",

    token_counter=model,

    include_system=True,

    allow_partial=False,

    start_on="human"
)

In [22]:
#RunnablePassthrough
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

In [23]:
chain = (
    RunnablePassthrough.assign(
        messages=itemgetter("messages") | trimmer
    )
    | prompt
    | model
)

Recommended Learning Order

Since you're aiming for AI/LLM engineering, I recommend mastering these concepts in this sequence:

ChatGroq / LLM invocation
HumanMessage, AIMessage, SystemMessage
ChatPromptTemplate
MessagesPlaceholder
LCEL (prompt | model)
ChatMessageHistory
RunnableWithMessageHistory
session_id
input_messages_key
trim_messages
RunnablePassthrough + itemgetter
Build a chatbot UI with Streamlit or FastAPI
Extend the chatbot into a Conversational RAG application
Add tools and convert it into an AI Agent